# Release16 Drawdown-Drift Short Training Screen

This notebook consumes the selected reward patch from `release16_drawdown_drift_reward_recipe_v1`, initializes each fold from the existing release16 fold-local trend BC target, runs a short PPO stage with the patched reward, and writes non-promotional evidence for the model-retention frontier.

Promotion and actor retention are blocked here by design; selected actors are represented only by fold-level metrics and daily path sidecars.

In [1]:

from __future__ import annotations

from dataclasses import asdict, replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import copy
import json
import math
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.dashboard import write_review_daily_csv
from src.soxl_quant.app.selection import train_with_selection
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.environment import build_initial_states
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'release16_drawdown_drift_short_training_screen_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
SOURCE_REWARD_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'release16_drawdown_drift_reward_recipe_v1'
SOURCE_TARGET_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'release16_wf2020_trend_target_bc_official_replay_v1'
TARGETS_PATH = SOURCE_TARGET_CARD / 'target_holdout_split.csv'
REWARD_PATCH_PATH = SOURCE_REWARD_CARD / 'selected_reward_patch.json'
REFERENCE_SELECTION_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / 'soxlbacktest_qlib_walkforward_baseline_v1' / 'selection.json'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260615
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 20.0
DRIFT_RELATIVE_FLOOR_PP = -5.0
DRIFT_MAX_DD_CEILING_PP = 2.0
BC_SLICE_ACCURACY_MIN = 0.55
REQUIRED_SLICES = {'release_gate_state': 3, 'is_downtrend_state': 5, 'drawdown_control_state': 3}
CARD.mkdir(parents=True, exist_ok=True)
DAILY_DIR = CARD / 'selected_daily_paths'
DAILY_DIR.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    'wf_2020': {
        'target_fold': 'wf_2020_train',
        'train': ('2010-03-11', '2018-12-31'),
        'valid': ('2019-01-01', '2019-12-31'),
        'test': ('2020-01-01', '2020-12-31'),
    },
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}

# Reuse the previously selected release16 BC recipe instead of doing another BC weight sweep.
BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=360, lr=0.00075, batch_size=128, seed=SEED + 23)
BC_WEIGHT_CONFIG = StateObjectiveBCSampleWeightConfig(
    rebound_multiplier=6.0,
    downtrend_multiplier=12.0,
    release_gate_multiplier=10.0,
    drawdown_control_multiplier=4.0,
    ordinary_multiplier=1.0,
    objective_weight_power=0.45,
    class_balanced_beta=0.94,
    weight_floor=0.05,
    weight_cap=70.0,
    normalize_mean=True,
)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def apply_reward_patch(profile, patch: dict[str, Any]):
    return replace(profile, reward=replace(profile.reward, **patch))


def build_replay_feature_frame(profile) -> pd.DataFrame:
    dataset = build_dataset(
        train_seg=('2010-03-11', '2022-12-31'),
        valid_seg=('2023-01-01', '2023-12-31'),
        test_seg=('2024-01-01', '2024-12-31'),
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    parts = [segment_frame(dataset, name) for name in ('train', 'valid', 'test')]
    frame = pd.concat(parts).sort_index()
    return frame.loc[~frame.index.duplicated(keep='last')].copy()

print('root', ROOT)
print('card', CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_drawdown_drift_short_training_screen_v1


In [2]:

source_reward_summary = json.loads((SOURCE_REWARD_CARD / 'summary.json').read_text(encoding='utf-8'))
source_reward_selection = json.loads((SOURCE_REWARD_CARD / 'selection.json').read_text(encoding='utf-8'))
source_target_summary = json.loads((SOURCE_TARGET_CARD / 'summary.json').read_text(encoding='utf-8'))
reward_patch = json.loads(REWARD_PATCH_PATH.read_text(encoding='utf-8'))
reference_selection = json.loads(REFERENCE_SELECTION_PATH.read_text(encoding='utf-8')) if REFERENCE_SELECTION_PATH.exists() else {}

base_profile = load_profile(PROFILE_NAME)
profile = apply_reward_patch(base_profile, reward_patch)
init_qlib(QLIB_DATA_PATH)
frame = build_replay_feature_frame(profile)
targets = pd.read_csv(TARGETS_PATH)
targets['datetime'] = pd.to_datetime(targets['datetime'])
for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state'):
    if col in targets.columns:
        targets[col] = targets[col].fillna(False).astype(bool)
    else:
        targets[col] = False

def fold_target_rows(fold_name: str, spec: dict[str, Any], split: str) -> pd.DataFrame:
    train_start = pd.Timestamp(spec['train'][0])
    train_end = pd.Timestamp(spec['train'][1])
    fold_all = targets[
        targets['wf_window'].astype(str).eq(fold_name)
        & targets['fold'].astype(str).eq(spec['target_fold'])
        & targets['bc_target_split'].astype(str).eq(split)
    ].copy()
    return fold_all[fold_all['datetime'].between(train_start, train_end)].copy()

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_targets = fold_target_rows(fold_name, spec, 'train')
    holdout_targets = fold_target_rows(fold_name, spec, 'holdout')
    fold_all = targets[
        targets['wf_window'].astype(str).eq(fold_name)
        & targets['fold'].astype(str).eq(spec['target_fold'])
    ].copy()
    valid_start, valid_end = pd.Timestamp(spec['valid'][0]), pd.Timestamp(spec['valid'][1])
    test_start, test_end = pd.Timestamp(spec['test'][0]), pd.Timestamp(spec['test'][1])
    coverage_rows.append({
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'train_rows': int(len(train_targets)),
        'holdout_rows': int(len(holdout_targets)),
        'raw_target_rows': int(len(fold_all)),
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
        'train_valid_overlap_rows': int(train_targets['datetime'].between(valid_start, valid_end).sum()),
        'train_test_overlap_rows': int(train_targets['datetime'].between(test_start, test_end).sum()),
        'holdout_test_overlap_rows': int(holdout_targets['datetime'].between(test_start, test_end).sum()),
        'train_release_gate_rows': int(train_targets['release_gate_state'].sum()),
        'train_downtrend_rows': int(train_targets['is_downtrend_state'].sum()),
        'train_drawdown_control_rows': int(train_targets['drawdown_control_state'].sum()),
        'holdout_release_gate_rows': int(holdout_targets['release_gate_state'].sum()),
        'holdout_downtrend_rows': int(holdout_targets['is_downtrend_state'].sum()),
        'holdout_drawdown_control_rows': int(holdout_targets['drawdown_control_state'].sum()),
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'source_reward_experiment': 'release16_drawdown_drift_reward_recipe_v1',
    'source_reward_decision': source_reward_summary.get('decision'),
    'source_target_experiment': 'release16_wf2020_trend_target_bc_official_replay_v1',
    'source_target_decision': source_target_summary.get('decision'),
    'reward_patch_path': str(REWARD_PATCH_PATH.relative_to(ROOT)),
    'target_rows_path': str(TARGETS_PATH.relative_to(ROOT)),
    'feature_frame_rows': int(len(frame)),
    'feature_frame_columns': int(len(frame.columns)),
    'rows': coverage.to_dict('records'),
})
print('source_reward_decision', source_reward_summary.get('decision'))
print('source_target_decision', source_target_summary.get('decision'))
print('reward_patch', reward_patch)
display(coverage)


[31537:MainThread](2026-06-15 15:39:11,276) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[31537:MainThread](2026-06-15 15:39:11,689) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[31537:MainThread](2026-06-15 15:39:11,689) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[31537:MainThread](2026-06-15 15:39:13,006) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[31537:MainThread](2026-06-15 15:39:13,006) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[31537:MainThread](2026-06-15 15:39:13,006) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


source_reward_decision release16_drawdown_drift_reward_recipe_ready_for_short_training_screen
source_target_decision release16_wf2020_trend_target_bc_slice_failed_no_promotion
reward_patch {'drawdown_drift_penalty': 1.5, 'drawdown_drift_position_threshold': 0.05, 'drawdown_drift_reward_enabled': True, 'drawdown_drift_threshold_pct': 18.0, 'trend_preservation_floor_enabled': True, 'trend_preservation_floor_penalty': 1.0, 'trend_preservation_floor_position': 0.25, 'trend_preservation_lookback': 21, 'trend_preservation_min_buyhold_return_pct': 5.0}


,fold,target_fold,train_rows,holdout_rows,raw_target_rows,train_start,train_end,valid_start,valid_end,test_start,test_end,train_valid_overlap_rows,train_test_overlap_rows,holdout_test_overlap_rows,train_release_gate_rows,train_downtrend_rows,train_drawdown_control_rows,holdout_release_gate_rows,holdout_downtrend_rows,holdout_drawdown_control_rows
0,wf_2020,wf_2020_train,1775,443,2470,2010-03-11,2018-12-31,2019-01-01,2019-12-31,2020-01-01,2020-12-31,0,0,0,24,760,765,14,192,193
1,wf_2022,wf_2022_train,200,53,253,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,0,0,0,5,96,18,3,24,6
2,wf_2023_2024,wf_2023_2024_train,600,156,756,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31,0,0,0,8,382,28,3,98,7


In [3]:

def actor_metrics_and_backtest(
    actor,
    eval_df: pd.DataFrame,
    price_series: pd.Series,
    profile,
    test_seg: tuple[str, str],
) -> tuple[dict[str, float], dict[str, Any]]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = dict(split_metrics)
    for src, dst in [
        ('return', 'return'),
        ('sharpe', 'sharpe'),
        ('max_dd', 'max_dd'),
        ('bnh_return', 'buy_and_hold_return'),
        ('relative_outperformance', 'relative_return'),
    ]:
        if src in bt:
            out[dst] = float(bt[src])
    return out, bt


def actor_metrics(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str]) -> dict[str, float]:
    metrics, _bt = actor_metrics_and_backtest(actor, eval_df, price_series, profile, test_seg)
    return metrics


def candidate_loss(row: dict[str, Any]) -> float:
    rel = float(row.get('test_relative_return', float('-inf')))
    dd = float(row.get('test_max_dd', float('inf')))
    rel_delta = float(row.get('relative_delta_vs_bc', float('-inf')))
    dd_delta = float(row.get('max_dd_delta_vs_bc', float('inf')))
    loss = 0.0
    loss += max(0.0, TARGET_RELATIVE_RETURN - rel) / max(TARGET_RELATIVE_RETURN, 1.0)
    loss += max(0.0, dd - TARGET_MAX_DD) / max(TARGET_MAX_DD, 1.0)
    loss += max(0.0, DRIFT_RELATIVE_FLOOR_PP - rel_delta) / max(abs(DRIFT_RELATIVE_FLOOR_PP), 1.0)
    loss += max(0.0, dd_delta - DRIFT_MAX_DD_CEILING_PP) / max(DRIFT_MAX_DD_CEILING_PP, 1.0)
    if not bool(row.get('train_slice_gate', False)):
        loss += 0.25
    if not bool(row.get('holdout_slice_gate', False)):
        loss += 0.25
    if row.get('candidate_kind') not in {'bc_initialization', 'ppo_retained'}:
        loss += 0.001 * max(float(row.get('iter', 0) or 0), 0.0)
    return float(loss)


def slice_gate_from_metrics(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, Any]]:
    indexed = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    details: dict[str, Any] = {}
    gate = True
    for slice_name, min_samples in REQUIRED_SLICES.items():
        if slice_name not in indexed.index:
            row = {'samples': 0, 'accuracy': float('nan'), 'gate': False}
        else:
            item = indexed.loc[slice_name]
            samples = int(item.get('samples', 0))
            accuracy = float(item.get('accuracy', float('nan')))
            row = {'samples': samples, 'accuracy': accuracy}
            row['gate'] = bool(samples >= min_samples and math.isfinite(accuracy) and accuracy >= BC_SLICE_ACCURACY_MIN)
        row['min_samples'] = int(min_samples)
        row['min_accuracy'] = float(BC_SLICE_ACCURACY_MIN)
        details[slice_name] = row
        gate = gate and bool(row['gate'])
    return bool(gate), details


In [4]:

obs_dim = observation_dim_for_profile(profile)

candidate_rows: list[dict[str, Any]] = []
selection_rows: list[dict[str, Any]] = []
history_rows: list[dict[str, Any]] = []
bc_fit_rows: list[dict[str, Any]] = []
bc_action_frames: list[pd.DataFrame] = []
bc_slice_frames: list[pd.DataFrame] = []

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    print(f"\n[fold:{fold_name}] target_fold={spec['target_fold']} train={spec['train']} valid={spec['valid']} test={spec['test']}")
    set_seed(SEED)
    train_targets = fold_target_rows(fold_name, spec, 'train')
    holdout_targets = fold_target_rows(fold_name, spec, 'holdout')
    if train_targets.empty:
        selection_rows.append({'fold': fold_name, 'status': 'skipped_no_train_targets', 'selected_candidate': None})
        continue

    bc_result = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=train_targets,
        profile=profile,
        train_config=BC_TRAIN_CONFIG,
        sample_weight_config=BC_WEIGHT_CONFIG,
        sample_size=0,
    )
    bc_fit_row = {'fold': fold_name, **bc_result.metrics}
    bc_fit_rows.append(bc_fit_row)
    action_dist = bc_result.action_distribution.copy()
    action_dist.insert(0, 'fold', fold_name)
    bc_action_frames.append(action_dist)

    train_dataset = build_state_objective_bc_dataset_from_frame(frame, targets=train_targets, profile=profile, sample_size=0)
    train_slice_metrics = evaluate_bc_actor_slice_fit(
        bc_result.actor,
        bc_result.obs,
        bc_result.actions,
        train_dataset.metadata,
        slice_cols=tuple(REQUIRED_SLICES),
    )
    train_slice_gate, train_slice_details = slice_gate_from_metrics(train_slice_metrics)
    tagged_train_slices = train_slice_metrics.copy()
    tagged_train_slices.insert(0, 'target_split', 'train')
    tagged_train_slices.insert(0, 'fold', fold_name)
    bc_slice_frames.append(tagged_train_slices)

    if holdout_targets.empty:
        holdout_slice_metrics = pd.DataFrame()
        holdout_slice_gate = False
        holdout_slice_details = {name: {'samples': 0, 'accuracy': float('nan'), 'gate': False} for name in REQUIRED_SLICES}
    else:
        holdout_dataset = build_state_objective_bc_dataset_from_frame(frame, targets=holdout_targets, profile=profile, sample_size=0)
        holdout_slice_metrics = evaluate_bc_actor_slice_fit(
            bc_result.actor,
            holdout_dataset.obs,
            holdout_dataset.actions,
            holdout_dataset.metadata,
            slice_cols=tuple(REQUIRED_SLICES),
        )
        holdout_slice_gate, holdout_slice_details = slice_gate_from_metrics(holdout_slice_metrics)
        tagged_holdout_slices = holdout_slice_metrics.copy()
        tagged_holdout_slices.insert(0, 'target_split', 'holdout')
        tagged_holdout_slices.insert(0, 'fold', fold_name)
        bc_slice_frames.append(tagged_holdout_slices)

    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, 'train')
    valid_df = segment_frame(dataset, 'valid')
    test_df = segment_frame(dataset, 'test')
    price_frame = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
    valid_price_series = price_frame.loc[spec['valid'][0]:spec['valid'][1]]
    test_price_series = price_frame.loc[spec['test'][0]:spec['test'][1]]

    train_pickle = CARD / f'{fold_name}_train_frame.pkl'
    train_df.to_pickle(train_pickle)
    initial_states = build_initial_states([(train_pickle, train_df, 0)], profile, smoke=False)
    if not initial_states:
        selection_rows.append({'fold': fold_name, 'status': 'skipped_no_initial_states', 'selected_candidate': None})
        continue

    policy, actor = build_policy(obs_dim, profile)
    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.eval()
    bc_valid_metrics = actor_metrics(actor, valid_df, valid_price_series, profile, spec['valid'])
    bc_test_metrics, bc_test_bt = actor_metrics_and_backtest(actor, test_df, test_price_series, profile, spec['test'])
    bc_relative = float(bc_test_metrics.get('relative_return', float('nan')))
    bc_max_dd = float(bc_test_metrics.get('max_dd', float('nan')))

    fold_state_cache: dict[str, dict[str, Any]] = {}

    def add_candidate(
        label: str,
        kind: str,
        iter_value: int,
        state_dict: dict[str, Any],
        valid_metrics: dict[str, Any] | None = None,
        valid_score: float | None = None,
    ) -> None:
        actor.load_state_dict(copy.deepcopy(state_dict))
        actor.eval()
        test_metrics, test_bt = actor_metrics_and_backtest(actor, test_df, test_price_series, profile, spec['test'])
        if valid_metrics is None:
            valid_metrics = actor_metrics(actor, valid_df, valid_price_series, profile, spec['valid'])
        rel = float(test_metrics.get('relative_return', float('nan')))
        dd = float(test_metrics.get('max_dd', float('nan')))
        rel_delta = rel - bc_relative
        dd_delta = dd - bc_max_dd
        daily_path = DAILY_DIR / f'{fold_name}__{label}_daily_path.csv'
        write_review_daily_csv(daily_path, test_bt['portfolio'], test_price_series, test_bt.get('position_history', []), test_bt.get('trade_signals', []))
        row = {
            'fold': fold_name,
            'candidate': label,
            'candidate_kind': kind,
            'iter': int(iter_value),
            'train_slice_gate': bool(train_slice_gate),
            'holdout_slice_gate': bool(holdout_slice_gate),
            'train_release_accuracy': float(train_slice_details['release_gate_state']['accuracy']),
            'train_downtrend_accuracy': float(train_slice_details['is_downtrend_state']['accuracy']),
            'train_drawdown_control_accuracy': float(train_slice_details['drawdown_control_state']['accuracy']),
            'holdout_release_accuracy': float(holdout_slice_details['release_gate_state']['accuracy']),
            'holdout_downtrend_accuracy': float(holdout_slice_details['is_downtrend_state']['accuracy']),
            'holdout_drawdown_control_accuracy': float(holdout_slice_details['drawdown_control_state']['accuracy']),
            'valid_score': None if valid_score is None else float(valid_score),
            'valid_return': float(valid_metrics.get('return', float('nan'))),
            'valid_sharpe': float(valid_metrics.get('sharpe', float('nan'))),
            'valid_max_dd': float(valid_metrics.get('max_dd', float('nan'))),
            'test_return': float(test_metrics.get('return', float('nan'))),
            'test_sharpe': float(test_metrics.get('sharpe', float('nan'))),
            'test_max_dd': dd,
            'test_avg_pos': float(test_metrics.get('avg_pos', float('nan'))),
            'test_buy_and_hold_return': float(test_metrics.get('buy_and_hold_return', float('nan'))),
            'test_relative_return': rel,
            'relative_delta_vs_bc': rel_delta,
            'max_dd_delta_vs_bc': dd_delta,
            'target_gate': bool(rel >= TARGET_RELATIVE_RETURN and dd < TARGET_MAX_DD),
            'drift_gate': bool(rel_delta >= DRIFT_RELATIVE_FLOOR_PP and dd_delta <= DRIFT_MAX_DD_CEILING_PP),
            'daily_path': str(daily_path.relative_to(ROOT)),
        }
        row['candidate_gate'] = bool(row['target_gate'] and row['drift_gate'] and row['train_slice_gate'] and row['holdout_slice_gate'])
        row['selection_loss'] = candidate_loss(row)
        candidate_rows.append(row)
        fold_state_cache[label] = copy.deepcopy(state_dict)

    add_candidate('bc_initialization', 'bc_initialization', 0, copy.deepcopy(bc_result.actor.state_dict()), bc_valid_metrics, None)

    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.train()
    selection = train_with_selection(
        policy=policy,
        actor=actor,
        initial_states=initial_states,
        obs_dim=obs_dim,
        profile=profile,
        valid_df=valid_df,
        smoke=False,
        test_df=None,
    )
    for item in selection.get('history', []):
        row = dict(item)
        row['fold'] = fold_name
        history_rows.append(row)

    seen_labels = {'bc_initialization'}
    for retained in selection.get('retained_checkpoints', []):
        iter_value = int(retained.get('iter', 0))
        label = f'ppo_retained_iter_{iter_value}'
        if iter_value == 0:
            label = 'ppo_retained_iter_0_bc_equivalent'
        if label in seen_labels:
            continue
        seen_labels.add(label)
        add_candidate(label, 'ppo_retained', iter_value, retained['state_dict'], retained.get('metrics'), retained.get('score'))

    add_candidate('ppo_selected_best', 'ppo_selected', int(profile.max_iters), copy.deepcopy(actor.state_dict()), selection.get('best_metrics'), selection.get('best_score'))

    fold_candidates = pd.DataFrame([row for row in candidate_rows if row['fold'] == fold_name])
    ranked = fold_candidates.sort_values(
        ['candidate_gate', 'selection_loss', 'candidate_kind'],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    selected = ranked.iloc[0].to_dict()
    selection_rows.append({
        'fold': fold_name,
        'status': 'evaluated',
        'selected_candidate': selected['candidate'],
        'selected_kind': selected['candidate_kind'],
        'selected_iter': int(selected['iter']),
        'selected_target_gate': bool(selected['target_gate']),
        'selected_drift_gate': bool(selected['drift_gate']),
        'selected_train_slice_gate': bool(selected['train_slice_gate']),
        'selected_holdout_slice_gate': bool(selected['holdout_slice_gate']),
        'selected_candidate_gate': bool(selected['candidate_gate']),
        'selected_loss': float(selected['selection_loss']),
        'selected_relative_return': float(selected['test_relative_return']),
        'selected_max_dd': float(selected['test_max_dd']),
        'selected_avg_pos': float(selected['test_avg_pos']),
        'selected_trade_count': int(pd.read_csv(ROOT / selected['daily_path'])['action'].astype(str).ne('HOLD').sum()),
        'bc_relative_return': bc_relative,
        'bc_max_dd': bc_max_dd,
        'relative_delta_vs_bc': float(selected['relative_delta_vs_bc']),
        'max_dd_delta_vs_bc': float(selected['max_dd_delta_vs_bc']),
        'candidate_count': int(len(ranked)),
        'selected_daily_path': selected['daily_path'],
    })

for train_pickle in CARD.glob('wf_*_train_frame.pkl'):
    train_pickle.unlink(missing_ok=True)

candidate_metrics = pd.DataFrame(candidate_rows).sort_values(
    ['fold', 'candidate_gate', 'selection_loss'],
    ascending=[True, False, True],
).reset_index(drop=True)
selection_by_fold = pd.DataFrame(selection_rows).sort_values(['fold']).reset_index(drop=True)
history = pd.DataFrame(history_rows)
bc_fit_metrics = pd.DataFrame(bc_fit_rows)
bc_action_distribution = pd.concat(bc_action_frames, ignore_index=True) if bc_action_frames else pd.DataFrame()
bc_slice_metrics_all = pd.concat(bc_slice_frames, ignore_index=True) if bc_slice_frames else pd.DataFrame()

candidate_metrics.to_csv(CARD / 'checkpoint_candidate_metrics.csv', index=False)
selection_by_fold.to_csv(CARD / 'checkpoint_selection_by_fold.csv', index=False)
history.to_csv(CARD / 'ppo_selection_history.csv', index=False)
bc_fit_metrics.to_csv(CARD / 'bc_fit_metrics.csv', index=False)
bc_action_distribution.to_csv(CARD / 'bc_action_distribution.csv', index=False)
bc_slice_metrics_all.to_csv(CARD / 'bc_slice_metrics.csv', index=False)

display(candidate_metrics)
display(selection_by_fold)



[fold:wf_2020] target_fold=wf_2020_train train=('2010-03-11', '2018-12-31') valid=('2019-01-01', '2019-12-31') test=('2020-01-01', '2020-12-31')


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[31537:MainThread](2026-06-15 15:39:18,782) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[31537:MainThread](2026-06-15 15:39:18,783) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[31537:MainThread](2026-06-15 15:39:18,783) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


...

[31537:MainThread](2026-06-15 15:42:00,415) WARNING - qlib.online operator - [exchange.py:228] - trade unit 1 is not supported in adjusted_price mode.


backtest loop:   0%|          | 0/251 [00:00<?, ?it/s]

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

,fold,candidate,candidate_kind,iter,train_slice_gate,holdout_slice_gate,train_release_accuracy,train_downtrend_accuracy,train_drawdown_control_accuracy,holdout_release_accuracy,...,test_avg_pos,test_buy_and_hold_return,test_relative_return,relative_delta_vs_bc,max_dd_delta_vs_bc,target_gate,drift_gate,daily_path,candidate_gate,selection_loss
0,wf_2020,ppo_retained_iter_4,ppo_retained,4,True,False,0.789474,0.856209,0.857374,0.461538,...,0.388424,333.333347,-262.992258,1.736636,0.018908,False,True,runtime/model_selection/card/release16_drawdow...,False,10.063285
1,wf_2020,ppo_selected_best,ppo_selected,4,True,False,0.789474,0.856209,0.857374,0.461538,...,0.388424,333.333347,-262.992258,1.736636,0.018908,False,True,runtime/model_selection/card/release16_drawdow...,False,10.067285
2,wf_2020,bc_initialization,bc_initialization,0,True,False,0.789474,0.856209,0.857374,0.461538,...,0.391214,333.333347,-264.728894,0.000000,0.000000,False,True,runtime/model_selection/card/release16_drawdow...,False,10.120228
3,wf_2020,ppo_retained_iter_0_bc_equivalent,ppo_retained,0,True,False,0.789474,0.856209,0.857374,0.461538,...,0.391214,333.333347,-264.728894,0.000000,0.000000,False,True,runtime/model_selection/card/release16_drawdow...,False,10.120228
4,wf_2022,bc_initialization,bc_initialization,0,False,False,0.000000,0.895833,0.944444,0.000000,...,0.162569,-78.573156,55.748206,0.000000,0.000000,False,True,runtime/model_selection/card/release16_drawdow...,False,0.841646
5,wf_2022,ppo_retained_iter_0_bc_equivalent,ppo_retained,0,False,False,0.000000,0.895833,0.944444,0.000000,...,0.162569,-78.573156,55.748206,0.000000,0.000000,False,True,runtime/model_selection/card/release16_drawdow...,False,0.841646
6,wf_2022,ppo_retained_iter_4,ppo_retained,4,False,False,0.000000,0.895833,0.944444,0.000000,...,0.182075,-78.573156,47.453799,-8.294406,10.564706,False,False,runtime/model_selection/card/release16_drawdow...,False,6.311115
7,wf_2022,ppo_selected_best,ppo_selected,4,False,False,0.000000,0.895833,0.944444,0.000000,...,0.182075,-78.573156,47.453799,-8.294406,10.564706,False,False,runtime/model_selection/card/release16_drawdow...,False,6.315115
8,wf_2023_2024,bc_initialization,bc_initialization,0,False,False,0.375000,0.672775,0.464286,0.333333,...,0.185929,-36.191701,32.133653,0.000000,0.000000,False,True,runtime/model_selection/card/release16_drawdow...,False,0.646200
9,wf_2023_2024,ppo_retained_iter_0_bc_equivalent,ppo_retained,0,False,False,0.375000,0.672775,0.464286,0.333333,...,0.185929,-36.191701,32.133653,0.000000,0.000000,False,True,runtime/model_selection/card/release16_drawdow...,False,0.646200


,fold,status,selected_candidate,selected_kind,selected_iter,selected_target_gate,selected_drift_gate,selected_train_slice_gate,selected_holdout_slice_gate,selected_candidate_gate,...,selected_relative_return,selected_max_dd,selected_avg_pos,selected_trade_count,bc_relative_return,bc_max_dd,relative_delta_vs_bc,max_dd_delta_vs_bc,candidate_count,selected_daily_path
0,wf_2020,evaluated,ppo_retained_iter_4,ppo_retained,4,False,True,True,False,False,...,-262.992258,20.937537,0.388424,33,-264.728894,20.918629,1.736636,0.018908,4,runtime/model_selection/card/release16_drawdow...
1,wf_2022,evaluated,bc_initialization,bc_initialization,0,False,True,False,False,False,...,55.748206,26.832912,0.162569,26,55.748206,26.832912,0.000000,0.000000,4,runtime/model_selection/card/release16_drawdow...
2,wf_2023_2024,evaluated,bc_initialization,bc_initialization,0,False,True,False,False,False,...,32.133653,22.924003,0.185929,28,32.133653,22.924003,0.000000,0.000000,4,runtime/model_selection/card/release16_drawdow...


In [5]:

selection_by_fold = pd.read_csv(CARD / 'checkpoint_selection_by_fold.csv')
candidate_metrics = pd.read_csv(CARD / 'checkpoint_candidate_metrics.csv')
required_folds = list(WALK_FORWARD_WINDOWS)
evaluated = selection_by_fold[selection_by_fold['status'].eq('evaluated')].copy()
required_fold_coverage = bool(set(required_folds).issubset(set(evaluated['fold'])) and len(evaluated) == len(required_folds))
all_candidate_gate = bool(required_fold_coverage and evaluated['selected_candidate_gate'].astype(bool).all())
any_candidate_gate = bool((not evaluated.empty) and evaluated['selected_candidate_gate'].astype(bool).any())
any_target_gate = bool((not evaluated.empty) and evaluated['selected_target_gate'].astype(bool).any())
any_ppo_selected = bool((not evaluated.empty) and evaluated['selected_kind'].astype(str).str.contains('ppo').any())
failed_folds = evaluated.loc[~evaluated['selected_candidate_gate'].astype(bool), 'fold'].tolist() if not evaluated.empty else required_folds
missing_folds = [fold for fold in required_folds if fold not in set(evaluated['fold'])]

if all_candidate_gate:
    decision = 'release16_drawdown_drift_short_training_screen_passed_no_promotion'
    next_mechanism = 'run_official_three_fold_replay_and_retention_gate_before_actor_retention'
elif any_candidate_gate:
    decision = 'release16_drawdown_drift_short_training_screen_partial_pass_no_promotion'
    next_mechanism = 'inspect_failed_fold_reward_drift_before_longer_training'
elif any_target_gate or any_ppo_selected:
    decision = 'release16_drawdown_drift_short_training_screen_mixed_no_promotion'
    next_mechanism = 'revise_reward_patch_or_checkpoint_selection_before_full_training'
else:
    decision = 'release16_drawdown_drift_short_training_screen_failed_no_promotion'
    next_mechanism = 'revise_drawdown_drift_reward_or_fold_targets_before_training'

selected_by_fold = {}
for row in evaluated.to_dict('records'):
    selected_by_fold[row['fold']] = {
        'selected_candidate': row['selected_candidate'],
        'selected_kind': row['selected_kind'],
        'selected_relative_return': float(row['selected_relative_return']),
        'selected_max_dd': float(row['selected_max_dd']),
        'selected_avg_pos': float(row['selected_avg_pos']),
        'selected_trade_count': int(row['selected_trade_count']),
        'selected_candidate_gate': bool(row['selected_candidate_gate']),
        'selected_target_gate': bool(row['selected_target_gate']),
        'selected_drift_gate': bool(row['selected_drift_gate']),
        'selected_train_slice_gate': bool(row['selected_train_slice_gate']),
        'selected_holdout_slice_gate': bool(row['selected_holdout_slice_gate']),
        'relative_delta_vs_bc': float(row['relative_delta_vs_bc']),
        'max_dd_delta_vs_bc': float(row['max_dd_delta_vs_bc']),
        'selected_daily_path': row['selected_daily_path'],
    }

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
    'direct_comparison_baseline': 'release16_drawdown_drift_reward_recipe_v1',
    'reference_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
    'role': 'diagnose_short_foldlocal_training',
    'profile': PROFILE_NAME,
    'changed_layer': ['reward', 'model_selection', 'bc_to_ppo_handoff'],
    'source_reward_recipe_experiment': 'release16_drawdown_drift_reward_recipe_v1',
    'source_reward_decision': source_reward_summary.get('decision'),
    'source_target_experiment': 'release16_wf2020_trend_target_bc_official_replay_v1',
    'source_target_decision': source_target_summary.get('decision'),
    'source_target_contract': str(TARGETS_PATH.relative_to(ROOT)),
    'reward_patch': reward_patch,
    'reward_patch_path': str(REWARD_PATCH_PATH.relative_to(ROOT)),
    'bc_train_config': asdict(BC_TRAIN_CONFIG),
    'bc_weight_config': asdict(BC_WEIGHT_CONFIG),
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'full_replay_allowed_next': bool(all_candidate_gate),
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'drawdown_gate_operator': '<',
    'drift_relative_floor_pp': DRIFT_RELATIVE_FLOOR_PP,
    'drift_max_dd_ceiling_pp': DRIFT_MAX_DD_CEILING_PP,
    'required_wf_windows': required_folds,
    'folds_evaluated': evaluated['fold'].tolist() if not evaluated.empty else [],
    'missing_folds': missing_folds,
    'failed_folds': failed_folds + missing_folds,
    'required_fold_coverage': required_fold_coverage,
    'selected_candidate_gate_count': int(evaluated['selected_candidate_gate'].sum()) if not evaluated.empty else 0,
    'selected_target_gate_count': int(evaluated['selected_target_gate'].sum()) if not evaluated.empty else 0,
    'selected_drift_gate_count': int(evaluated['selected_drift_gate'].sum()) if not evaluated.empty else 0,
    'selected_train_slice_gate_count': int(evaluated['selected_train_slice_gate'].sum()) if not evaluated.empty else 0,
    'selected_holdout_slice_gate_count': int(evaluated['selected_holdout_slice_gate'].sum()) if not evaluated.empty else 0,
    'selected_by_fold': selected_by_fold,
    'promotion_blocker': 'Short fold-local training screen only; no actor bundle is persisted or retained. Official three-fold replay, YTD return alignment, and strict retention gate are still required.',
    'sidecars': [
        'coverage.csv',
        'coverage.json',
        'bc_fit_metrics.csv',
        'bc_action_distribution.csv',
        'bc_slice_metrics.csv',
        'ppo_selection_history.csv',
        'checkpoint_candidate_metrics.csv',
        'checkpoint_selection_by_fold.csv',
        'selected_daily_paths/*.csv',
        'selection.json',
        'summary.json',
    ],
}
summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'next_mechanism': next_mechanism,
    'created_at': selection['created_at'],
    'canonical_baseline': selection['canonical_baseline'],
    'direct_comparison_baseline': selection['direct_comparison_baseline'],
    'reference_baseline': selection['reference_baseline'],
    'source_experiment': 'release16_drawdown_drift_reward_recipe_v1',
    'source_reward_recipe_experiment': 'release16_drawdown_drift_reward_recipe_v1',
    'source_reward_decision': source_reward_summary.get('decision'),
    'source_target_experiment': 'release16_wf2020_trend_target_bc_official_replay_v1',
    'source_target_decision': source_target_summary.get('decision'),
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'required_wf_windows': required_folds,
    'selected_by_fold': selected_by_fold,
    'worst_wf_max_drawdown_pct': float(evaluated['selected_max_dd'].max()) if not evaluated.empty else None,
    'min_wf_relative_return_pp': float(evaluated['selected_relative_return'].min()) if not evaluated.empty else None,
    'full_replay_allowed_next': bool(all_candidate_gate),
    'training_allowed_next': bool(all_candidate_gate),
    'primary_blocker': 'official_replay_required' if all_candidate_gate else 'short_training_gate_failed',
    'selection': selection,
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
selection


{'experiment': 'release16_drawdown_drift_short_training_screen_v1',
 'created_at': '2026-06-15T07:42:00.621433+00:00',
 'decision': 'release16_drawdown_drift_short_training_screen_mixed_no_promotion',
 'next_mechanism': 'revise_reward_patch_or_checkpoint_selection_before_full_training',
 'canonical_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'direct_comparison_baseline': 'release16_drawdown_drift_reward_recipe_v1',
 'reference_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'role': 'diagnose_short_foldlocal_training',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'changed_layer': ['reward', 'model_selection', 'bc_to_ppo_handoff'],
 'source_reward_recipe_experiment': 'release16_drawdown_drift_reward_recipe_v1',
 'source_reward_decision': 'release16_drawdown_drift_reward_recipe_ready_for_short_training_screen',
 'source_target_experiment': 'release16_wf2020_trend_target_bc_official_replay_v1',
 'source_target_decision': 'release16_wf2020_trend_target_bc_slice_fail

In [6]:

selection = json.loads((CARD / 'selection.json').read_text(encoding='utf-8'))
selection_by_fold = pd.read_csv(CARD / 'checkpoint_selection_by_fold.csv')
lines = [
    '## Conclusion',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
    f"Reward patch: `{selection['reward_patch_path']}`.",
    '',
    f"Gate: relative return >= {selection['target_relative_return_pp']:.0f}pp and MaxDD {selection['drawdown_gate_operator']} {selection['target_max_dd_pct']:.0f}% on `wf_2020`, `wf_2022`, and `wf_2023_2024`, plus non-negative short-training drift versus BC within configured tolerance.",
    '',
    'Selected candidates:',
]
for row in selection_by_fold.itertuples(index=False):
    if row.status != 'evaluated':
        lines.append(f'- {row.fold}: {row.status}')
        continue
    lines.append(
        f"- {row.fold}: `{row.selected_candidate}` ({row.selected_kind}), rel {row.selected_relative_return:.2f}pp, "
        f"MaxDD {row.selected_max_dd:.2f}%, avg_pos {row.selected_avg_pos:.3f}, "
        f"gate {bool(row.selected_candidate_gate)}, daily `{row.selected_daily_path}`"
    )
lines.extend([
    '',
    'Risk note: this card does not retain or promote any actor. It only tests whether the selected drawdown-drift reward patch survives a short fold-local PPO screen after release16 BC initialization.',
    '',
    f"Next action: `{selection['next_mechanism']}`.",
    '',
    'Artifacts: `checkpoint_candidate_metrics.csv`, `checkpoint_selection_by_fold.csv`, `bc_slice_metrics.csv`, `ppo_selection_history.csv`, `selected_daily_paths/*.csv`, `selection.json`, `summary.json`.',
])
conclusion = '\n'.join(lines).rstrip() + '\n'
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
display(Markdown(conclusion))


## Conclusion

Decision: `release16_drawdown_drift_short_training_screen_mixed_no_promotion`.

Reward patch: `runtime/model_selection/card/release16_drawdown_drift_reward_recipe_v1/selected_reward_patch.json`.

Gate: relative return >= 30pp and MaxDD < 20% on `wf_2020`, `wf_2022`, and `wf_2023_2024`, plus non-negative short-training drift versus BC within configured tolerance.

Selected candidates:
- wf_2020: `ppo_retained_iter_4` (ppo_retained), rel -262.99pp, MaxDD 20.94%, avg_pos 0.388, gate False, daily `runtime/model_selection/card/release16_drawdown_drift_short_training_screen_v1/selected_daily_paths/wf_2020__ppo_retained_iter_4_daily_path.csv`
- wf_2022: `bc_initialization` (bc_initialization), rel 55.75pp, MaxDD 26.83%, avg_pos 0.163, gate False, daily `runtime/model_selection/card/release16_drawdown_drift_short_training_screen_v1/selected_daily_paths/wf_2022__bc_initialization_daily_path.csv`
- wf_2023_2024: `bc_initialization` (bc_initialization), rel 32.13pp, MaxDD 22.92%, avg_pos 0.186, gate False, daily `runtime/model_selection/card/release16_drawdown_drift_short_training_screen_v1/selected_daily_paths/wf_2023_2024__bc_initialization_daily_path.csv`

Risk note: this card does not retain or promote any actor. It only tests whether the selected drawdown-drift reward patch survives a short fold-local PPO screen after release16 BC initialization.

Next action: `revise_reward_patch_or_checkpoint_selection_before_full_training`.

Artifacts: `checkpoint_candidate_metrics.csv`, `checkpoint_selection_by_fold.csv`, `bc_slice_metrics.csv`, `ppo_selection_history.csv`, `selected_daily_paths/*.csv`, `selection.json`, `summary.json`.
